# Embedding dataset compression & decompression workflow

This notebook is part of a two-step workflow used to keep the repository smaller while preserving compatibility with existing PyTorch training and analysis notebooks.

The original embedding dataset contains many repeated copies of the same `bacterium_embedding` and `phage_embedding` tensors across rows. This happens because the same `bacterium_id` or `phage_id` can appear multiple times in the interaction table. Saving the dataset directly as a `.pt` file duplicates those tensors many times and can make the file unnecessarily large. *(103Mb, while max alowed git push size is 100Mb...)*

## How the workflow works

### 1. Decomprassion part (most likely the one that you need)

The restoration step reconstructs the original DataFrame structure expected by the existing notebooks. **Run this part before interacting with the testing notebooks `nb1`, `nb2` & `3`**

It loads the compact file from `pbi/XAI/bundle` folder *(that is pulled with the project)*, maps embeddings back to each row using `bacterium_id` and `phage_id`, and saves the result under the original file name `test_dataset.pt`. This allows the rest of the project to continue using the same training code without major refactoring.

### 2. Compression part (the one that used to push dataset into GitHub)

The compression step creates a compact dataset file by:

- storing each unique bacterium embedding only once;
- storing each unique phage embedding only once;
- keeping the interaction table as lightweight metadata with IDs and labels.

The result is a file such as `compressedEmbData.pt`, which is much smaller than the original dataset after the deduplication.

### 3. Testing part (to double check that everything works properly)

# restore is first


In [12]:
# Decompress embedding dataset back to original DataFrame
# -------------------------------------------------------
from pathlib import Path
import torch
import pandas as pd

# Path to compressed file (produced by the script above)
compressed_path = Path("../bundle/compressedEmbData.pt")   # change if needed

# Original name expected by your notebooks (config.test_path)
# restored_path = Path("emb_test_data.pt")         # or Path(config.test_path).name
restored_path = compressed_path.with_name("test_dataset.pt")

# If you want tensors moved to a device immediately, set this:
restore_to_device = None          # e.g. "cuda:0" or "cuda:3"
# restore_to_device = "cuda:0"

assert compressed_path.exists(), f"File not found: {compressed_path.resolve()}"
print(f"Compressed input: {compressed_path.resolve()}")
print(f"Restored output:  {restored_path.resolve()}")

Compressed input: /data/pavel.degterev/pbi/XAI/bundle/compressedEmbData.pt
Restored output:  /data/pavel.degterev/pbi/XAI/bundle/test_dataset.pt


In [13]:
# --- Load compressed representation ---
compressed = torch.load(compressed_path, map_location="cpu")

required_keys = {"interactions", "bacterium_embeddings", "phage_embeddings"}
missing = required_keys - set(compressed.keys())
if missing:
    raise ValueError(f"Missing required keys: {missing}")

interactions = compressed["interactions"].copy().reset_index(drop=True)
bacterium_embeddings = compressed["bacterium_embeddings"]
phage_embeddings = compressed["phage_embeddings"]

In [14]:
# --- Rebuild original-style DataFrame ---
emb_test_data = interactions.copy()

emb_test_data["bacterium_embedding"] = emb_test_data["bacterium_id"].map(
    bacterium_embeddings
)
emb_test_data["phage_embedding"] = emb_test_data["phage_id"].map(
    phage_embeddings
)

if restore_to_device is not None:
    emb_test_data["bacterium_embedding"] = emb_test_data[
        "bacterium_embedding"
    ].map(lambda t: t.to(restore_to_device))
    emb_test_data["phage_embedding"] = emb_test_data[
        "phage_embedding"
    ].map(lambda t: t.to(restore_to_device))

# Ensure column order matches your existing code
emb_test_data = emb_test_data[
    [
        "id",
        "phage_id",
        "bacterium_id",
        "interaction_type",
        "bacterium_embedding",
        "phage_embedding",
    ]
]

torch.save(emb_test_data, restored_path)

print("Restored DataFrame saved successfully.")
print(f"Rows: {len(emb_test_data)}")
print(f"Columns: {list(emb_test_data.columns)}")

Restored DataFrame saved successfully.
Rows: 1426
Columns: ['id', 'phage_id', 'bacterium_id', 'interaction_type', 'bacterium_embedding', 'phage_embedding']


In [15]:
# --- Optional: compatibility check with your current notebook usage ---
emb_test_data_check = torch.load(restored_path, map_location="cpu")
emb_test_data_check = emb_test_data_check.reset_index(drop=True)

print("\n=== test initially designed to verify if right (filtered) df is used ===")
print(
    f"emb_test_data: {len(emb_test_data_check)} rows, expected value 1426 after cleaning"
)
print(
    "emb_test_data: "
    f"{len(emb_test_data_check['phage_embedding'].iloc[0])} "
    "embedding lenght for bacteria, expected value 14788 before pca"
)
print(
    f"\nDistribution for "
    f"{emb_test_data_check['interaction_type'].value_counts()}"
)


=== test initially designed to verify if right (filtered) df is used ===
emb_test_data: 1426 rows, expected value 1426 after cleaning
emb_test_data: 14788 embedding lenght for bacteria, expected value 14788 before pca

Distribution for interaction_type
0    1051
1     375
Name: count, dtype: int64


# save - second (most likely only restore will be needed)

In [7]:
# Compress embedding dataset: deduplicate embeddings by ID
# -------------------------------------------------------
# Configuration
from pathlib import Path
import torch
import pandas as pd

# Path to your current big test dataset
input_path = Path("../../output/best_model_XAI/test_dataset.pt")

# New compact file name to be committed to git
#output_path = input_path.with_name("compressedEmbData.pt")
output_path = Path("../bundle/compressedEmbData.pt")

# Optional: store embeddings as float16 to save extra space
USE_FLOAT16 = False

assert input_path.exists(), f"File not found: {input_path.resolve()}"
print(f"Input:  {input_path.resolve()}")
print(f"Output: {output_path.resolve()}")

Input:  /data/pavel.degterev/pbi/output/best_model_XAI/test_dataset.pt
Output: /data/pavel.degterev/pbi/XAI/bundle/compressedEmbData.pt


In [8]:
# --- Load original dataframe ---
emb_test_data = torch.load(input_path, map_location="cpu")

if not isinstance(emb_test_data, pd.DataFrame):
    raise TypeError(f"Expected pandas DataFrame, got {type(emb_test_data)}")

emb_test_data = emb_test_data.reset_index(drop=True)

required_cols = {
    "id",
    "phage_id",
    "bacterium_id",
    "interaction_type",
    "bacterium_embedding",
    "phage_embedding",
}
missing = required_cols - set(emb_test_data.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Loaded DataFrame")
print(f"Rows: {len(emb_test_data)}")
print(f"Unique bacterium_id: {emb_test_data['bacterium_id'].nunique()}")
print(f"Unique phage_id: {emb_test_data['phage_id'].nunique()}")

Loaded DataFrame
Rows: 1426
Unique bacterium_id: 166
Unique phage_id: 860


In [9]:
# --- Build compact representation ---
def to_cpu_tensor(x):
    if isinstance(x, torch.Tensor):
        t = x.detach().cpu()
    elif isinstance(x, (list, tuple)):
        t = torch.tensor(x)
    else:
        t = torch.as_tensor(x)
    if USE_FLOAT16 and torch.is_floating_point(t):
        t = t.half()
    return t


# Unique embedding per bacterium
bact_unique = (
    emb_test_data[["bacterium_id", "bacterium_embedding"]]
    .drop_duplicates("bacterium_id")
)

# Unique embedding per phage
phage_unique = (
    emb_test_data[["phage_id", "phage_embedding"]]
    .drop_duplicates("phage_id")
)

bacterium_embeddings = {
    int(row.bacterium_id): to_cpu_tensor(row.bacterium_embedding)
    for row in bact_unique.itertuples(index=False)
}

phage_embeddings = {
    int(row.phage_id): to_cpu_tensor(row.phage_embedding)
    for row in phage_unique.itertuples(index=False)
}

# Lightweight interaction table (no tensors)
interactions = (
    emb_test_data[["id", "phage_id", "bacterium_id", "interaction_type"]]
    .copy()
    .reset_index(drop=True)
)

compressed = {
    "format_version": 1,
    "source_filename": input_path.name,
    "interactions": interactions,
    "bacterium_embeddings": bacterium_embeddings,
    "phage_embeddings": phage_embeddings,
    "options": {"float16": USE_FLOAT16},
}

# --- Save compact file ---
torch.save(compressed, output_path)

orig_size = input_path.stat().st_size / (1024 ** 2)
new_size = output_path.stat().st_size / (1024 ** 2)
ratio = (new_size / orig_size) if orig_size else 0

print(f"Original size:   {orig_size:.2f} MB")
print(f"Compressed size: {new_size:.2f} MB")
print(f"Size ratio:      {ratio:.3f}")
print("Saved compressed dataset successfully.")

Original size:   103.19 MB
Compressed size: 51.37 MB
Size ratio:      0.498
Saved compressed dataset successfully.


In [10]:
# --- Quick validation preview ---
loaded = torch.load(output_path, map_location="cpu")

restored_preview = loaded["interactions"].copy()
restored_preview["bacterium_embedding"] = restored_preview["bacterium_id"].map(
    loaded["bacterium_embeddings"]
)
restored_preview["phage_embedding"] = restored_preview["phage_id"].map(
    loaded["phage_embeddings"]
)

print("\n=== validation preview ===")
print(f"rows: {len(restored_preview)}")
print(
    "embedding length example:",
    len(restored_preview["phage_embedding"].iloc[0]),
)
print(restored_preview["interaction_type"].value_counts())


=== validation preview ===
rows: 1426
embedding length example: 14788
interaction_type
0    1051
1     375
Name: count, dtype: int64


# similarity test

In [3]:
import torch
import pandas as pd
from pathlib import Path

orig_path = Path("../../output/best_model_XAI/test_dataset.pt")          # original
restored_path = Path("../bundle/test_dataset.pt")      # same name after decompression
# if you decompressed to a different name, set it here

assert orig_path.exists(), f"Missing original: {orig_path}"
assert restored_path.exists(), f"Missing restored: {restored_path}"

orig = torch.load(orig_path, map_location="cpu").reset_index(drop=True)
rest = torch.load(restored_path, map_location="cpu").reset_index(drop=True)

# 1) basic shape / column checks
print("=== shape / schema check ===")
print("orig rows:", len(orig), "restored rows:", len(rest))
print("orig cols:", list(orig.columns))
print("rest cols:", list(rest.columns))

assert list(orig.columns) == list(rest.columns), "Column sets differ"
assert len(orig) == len(rest), "Row counts differ"

# 2) non‑tensor columns equality
meta_cols = ["id", "phage_id", "bacterium_id", "interaction_type"]
pd.testing.assert_series_equal(
    orig[meta_cols].reset_index(drop=True).stack().sort_index(),
    rest[meta_cols].reset_index(drop=True).stack().sort_index(),
)
print("Non‑tensor columns are identical.")

# 3) tensor value equality (exact)
def tensors_equal(a, b):
    if type(a) is not type(b):
        return False
    if isinstance(a, torch.Tensor):
        return torch.equal(a, b)
    # fallback for lists/tuples of tensors
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(tensors_equal(x, y) for x, y in zip(a, b))
    return a == b

diff_count = 0
for col in ["bacterium_embedding", "phage_embedding"]:
    for i, (x, y) in enumerate(zip(orig[col], rest[col])):
        if not tensors_equal(x, y):
            diff_count += 1
            # print first few mismatches, then stop being noisy
            if diff_count <= 5:
                print(f"[{col}] mismatch at row {i}")
        #print(f"[{col}] match at row {i} with x {x} and y {y}")
    print(f"{col}: {len(orig)} rows checked")

if diff_count == 0:
    print("\n✅ Round‑trip OK: original and restored datasets are bit‑identical.")
else:
    print(f"\n⚠️ Found {diff_count} tensor mismatches between original and restored.")

=== shape / schema check ===
orig rows: 1426 restored rows: 1426
orig cols: ['id', 'phage_id', 'bacterium_id', 'interaction_type', 'bacterium_embedding', 'phage_embedding']
rest cols: ['id', 'phage_id', 'bacterium_id', 'interaction_type', 'bacterium_embedding', 'phage_embedding']
Non‑tensor columns are identical.
bacterium_embedding: 1426 rows checked
phage_embedding: 1426 rows checked

✅ Round‑trip OK: original and restored datasets are bit‑identical.
